# Neumann Sparsification — Grid Search

Systematic comparison of **Neumann Series**, **MBBr**, and **EffR** sparsification
across 56 configuration model variants (8 degree × 7 weight distributions).

All three methods keep the same number of edges (= MBB edge count) for fair comparison.
Evaluation: MSE of per-node infection probabilities from SIR Monte Carlo.

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_path = os.path.join(repo_root, 'src', 'python')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import time

from graph_sparsification.generators import configuration_model
from graph_sparsification.sparsifiers import (
    metric_backbone,
    metric_backbone_rescaled,
    effective_resistance_sparsify,
    to_proximity,
)
from graph_sparsification.sir import sir_monte_carlo, calibrate_beta
from graph_sparsification.neumann_sparsifier import neumann_sparsify

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

try:
    from graph_sparsification._sir_cpp import sir_simulation_cpp
    print('C++ SIR backend: available')
except ImportError:
    print('C++ SIR backend: NOT available (using Python fallback)')

## 1. Distribution definitions

In [ ]:
degree_distributions = {
    'Unif(1,50)':      lambda n, rng: rng.integers(1, 51, size=n),
    'Unif(1,100)':     lambda n, rng: rng.integers(1, 101, size=n),
    'Exp(30)':         lambda n, rng: np.ceil(rng.exponential(30, size=n)).astype(int),
    'Exp(60)':         lambda n, rng: np.ceil(rng.exponential(60, size=n)).astype(int),
    'LogN(3.26,0.66)': lambda n, rng: np.ceil(rng.lognormal(3.26, 0.66, size=n)).astype(int),
    'LogN(3.26,2)':    lambda n, rng: np.ceil(rng.lognormal(3.26, 2.0, size=n)).astype(int),
    'Pareto(2.5,20)':  lambda n, rng: np.ceil((rng.pareto(2.5, size=n) + 1) * 20).astype(int),
    'Pareto(1.5,30)':  lambda n, rng: np.ceil((rng.pareto(1.5, size=n) + 1) * 30).astype(int),
}

weight_distributions = {
    'Exp(1/30)':        lambda m, rng: rng.exponential(30.0, size=m),
    'Exp(1)':           lambda m, rng: rng.exponential(1.0, size=m),
    'Exp(30)':          lambda m, rng: rng.exponential(1/30, size=m),
    'LogN(2,1)':        lambda m, rng: rng.lognormal(2.0, 1.0, size=m),
    'LogLogN(1.2,0.4)': lambda m, rng: np.exp(rng.lognormal(1.2, 0.4, size=m)),
    'LogLogN(1.2,0.8)': lambda m, rng: np.exp(rng.lognormal(1.2, 0.8, size=m)),
    'LogLogN(2,0.8)':   lambda m, rng: np.exp(rng.lognormal(2.0, 0.8, size=m)),
}

print(f'{len(degree_distributions)} degree x {len(weight_distributions)} weight = '
      f'{len(degree_distributions) * len(weight_distributions)} configurations')

## 2. Experiment parameters

In [ ]:
N_NODES = 500
N_SIR_RUNS = 200
N_CAL_RUNS = 30
GAMMA = 1.0
SEED = 42

## 3. Main loop

In [ ]:
run_seed = 42
all_results = []

for deg_name, deg_sampler in degree_distributions.items():
    for wt_name, wt_sampler in weight_distributions.items():
        label = f'{deg_name}  |  {wt_name}'
        rng = np.random.default_rng(run_seed)

        # ── Generate graph ─────────────────────────────────────────
        W_dist = configuration_model(N_NODES, deg_sampler, wt_sampler, rng=rng)
        n_edges = sparse.triu(W_dist).nnz

        if n_edges < 10:
            print(f'SKIP {label}: only {n_edges} edges')
            continue

        W_prox = to_proximity(W_dist)

        print(f'\n{"="*72}')
        print(f'{label}  ({n_edges} edges)')

        # ── Calibrate beta ─────────────────────────────────────────
        beta, cal_info = calibrate_beta(
            W_prox, gamma=GAMMA,
            n_calibration_runs=N_CAL_RUNS, rng=rng, verbose=False,
        )
        print(f'  beta={beta:.4f}, mean_inf={cal_info["mean_infection"]:.3f}')

        # ── Sparsify ───────────────────────────────────────────────
        W_mbb_dist = metric_backbone(W_dist)
        n_mbb = sparse.triu(W_mbb_dist).nnz
        retention = n_mbb / n_edges

        W_mbbr = metric_backbone_rescaled(W_dist)

        W_effr = effective_resistance_sparsify(
            W_prox, n_edges=n_mbb, rng=np.random.default_rng(SEED))

        t0 = time.time()
        W_neumann = neumann_sparsify(
            W_dist, n_mbb, seed=SEED, verbose=False)
        t_neur = time.time() - t0

        print(f'  MBB edges: {n_mbb} ({100*retention:.1f}%), '
              f'Neumann: {t_neur:.0f}s')

        # ── SIR ────────────────────────────────────────────────────
        initial = [int(np.argmax(np.array(W_prox.sum(axis=1)).ravel()))]

        sir_orig = sir_monte_carlo(
            W_prox, beta, GAMMA, initial,
            n_runs=N_SIR_RUNS, rng=np.random.default_rng(100))
        sir_mbbr = sir_monte_carlo(
            W_mbbr, beta, GAMMA, initial,
            n_runs=N_SIR_RUNS, rng=np.random.default_rng(100))
        sir_effr = sir_monte_carlo(
            W_effr, beta, GAMMA, initial,
            n_runs=N_SIR_RUNS, rng=np.random.default_rng(100))
        sir_neur = sir_monte_carlo(
            W_neumann, beta, GAMMA, initial,
            n_runs=N_SIR_RUNS, rng=np.random.default_rng(100))

        p_orig = sir_orig['infection_prob']

        def _mse(po, ps):
            m = np.isfinite(po) & np.isfinite(ps)
            return np.mean((po[m] - ps[m])**2)

        mse_mbbr = _mse(p_orig, sir_mbbr['infection_prob'])
        mse_effr = _mse(p_orig, sir_effr['infection_prob'])
        mse_neur = _mse(p_orig, sir_neur['infection_prob'])

        best = min(mse_mbbr, mse_effr)
        status = 'WIN' if mse_neur <= best else f'{mse_neur/best:.2f}x'
        print(f'  MBBr={mse_mbbr:.6f}  EffR={mse_effr:.6f}  '
              f'Neumann={mse_neur:.6f}  [{status}]')

        all_results.append({
            'deg': deg_name, 'wt': wt_name, 'label': label,
            'n_edges': n_edges, 'n_mbb': n_mbb,
            'retention': retention, 'beta': beta,
            'mse_mbbr': mse_mbbr, 'mse_effr': mse_effr, 'mse_neur': mse_neur,
        })

print(f'\n\nDone: {len(all_results)} configurations')

## 4. Summary table

In [ ]:
print(f"{'Degree':<18} {'Weights':<18} {'Ret%':>5} {'MBBr':>10} {'EffR':>10} "
      f"{'Neumann':>10} {'Status':>8}")
print('-' * 85)

n_wins = 0
n_beat_effr = 0
n_beat_mbbr = 0

for r in all_results:
    best = min(r['mse_mbbr'], r['mse_effr'])
    status = 'WIN' if r['mse_neur'] <= best else f"{r['mse_neur']/best:.2f}x"
    if r['mse_neur'] <= best:
        n_wins += 1
    if r['mse_neur'] <= r['mse_effr']:
        n_beat_effr += 1
    if r['mse_neur'] <= r['mse_mbbr']:
        n_beat_mbbr += 1
    print(f"{r['deg']:<18} {r['wt']:<18} {r['retention']*100:>4.1f}% "
          f"{r['mse_mbbr']:>10.6f} {r['mse_effr']:>10.6f} "
          f"{r['mse_neur']:>10.6f} {status:>8}")

n = len(all_results)
print(f'\nWins vs best: {n_wins}/{n}')
print(f'Beats EffR:   {n_beat_effr}/{n}')
print(f'Beats MBBr:   {n_beat_mbbr}/{n}')

## 5. Heatmap utilities

In [ ]:
deg_names = list(degree_distributions.keys())
wt_names  = list(weight_distributions.keys())
n_deg, n_wt = len(deg_names), len(wt_names)

res_lookup = {(r['deg'], r['wt']): r for r in all_results}

def build_grid(key):
    grid = np.full((n_deg, n_wt), np.nan)
    for i, d in enumerate(deg_names):
        for j, w in enumerate(wt_names):
            if (d, w) in res_lookup:
                grid[i, j] = res_lookup[(d, w)][key]
    return grid

def plot_heatmap(grid, title, cmap='viridis', fmt='.4f', vmin=None, vmax=None,
                 annot_color=None, figsize=None):
    if figsize is None:
        figsize = (max(9, n_wt * 1.2), max(5, n_deg * 0.75))
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(grid, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(n_wt))
    ax.set_xticklabels(wt_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(n_deg))
    ax.set_yticklabels(deg_names, fontsize=9)
    ax.set_xlabel('Weight distribution', fontsize=11)
    ax.set_ylabel('Degree distribution', fontsize=11)
    ax.set_title(title, fontsize=13, pad=10)
    for i in range(n_deg):
        for j in range(n_wt):
            val = grid[i, j]
            if np.isfinite(val):
                color = annot_color or ('white' if im.norm(val) < 0.5 else 'black')
                ax.text(j, i, f'{val:{fmt}}', ha='center', va='center',
                        fontsize=7, color=color)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    return fig

## 6. MSE heatmaps — all three methods

In [ ]:
grid_mse_mbbr = build_grid('mse_mbbr')
grid_mse_effr = build_grid('mse_effr')
grid_mse_neur = build_grid('mse_neur')

# Shared color scale
all_mse = np.concatenate([grid_mse_mbbr.ravel(), grid_mse_effr.ravel(),
                          grid_mse_neur.ravel()])
mse_max = np.nanpercentile(all_mse, 95)  # clip outliers for better contrast

for grid, name in [(grid_mse_mbbr, 'MBBr'),
                    (grid_mse_effr, 'Eff. Resistance'),
                    (grid_mse_neur, 'Neumann')]:
    fig = plot_heatmap(grid, f'MSE of infection prob. — {name}',
                       cmap='magma_r', fmt='.4f', vmin=0, vmax=mse_max)
    plt.show()

## 7. MSE difference heatmaps

In [ ]:
# Neumann - MBBr: negative = Neumann better
grid_diff_mbbr = grid_mse_neur - grid_mse_mbbr
vabs = np.nanpercentile(np.abs(grid_diff_mbbr), 95)
fig = plot_heatmap(
    grid_diff_mbbr,
    'MSE difference (Neumann \u2212 MBBr)\nnegative (blue) = Neumann better',
    cmap='RdBu', fmt='.4f', vmin=-vabs, vmax=vabs, annot_color='black',
)
plt.show()

# Neumann - EffR: negative = Neumann better
grid_diff_effr = grid_mse_neur - grid_mse_effr
vabs = np.nanpercentile(np.abs(grid_diff_effr), 95)
fig = plot_heatmap(
    grid_diff_effr,
    'MSE difference (Neumann \u2212 EffR)\nnegative (blue) = Neumann better',
    cmap='RdBu', fmt='.4f', vmin=-vabs, vmax=vabs, annot_color='black',
)
plt.show()

## 8. Win/loss summary

In [ ]:
# Win grid: 1 = Neumann beats best, 0 = loses, 0.5 = within 10%
def build_status_grid():
    grid = np.full((n_deg, n_wt), np.nan)
    for i, d in enumerate(deg_names):
        for j, w in enumerate(wt_names):
            if (d, w) in res_lookup:
                r = res_lookup[(d, w)]
                best = min(r['mse_mbbr'], r['mse_effr'])
                if r['mse_neur'] <= best:
                    grid[i, j] = 1.0   # WIN
                elif r['mse_neur'] <= best * 1.1:
                    grid[i, j] = 0.5   # close
                else:
                    grid[i, j] = 0.0   # lose
    return grid

grid_status = build_status_grid()
fig = plot_heatmap(
    grid_status,
    'Neumann vs best baseline\n1.0 = WIN, 0.5 = within 10%, 0.0 = lose',
    cmap='RdYlGn', fmt='.1f', vmin=0, vmax=1, annot_color='black',
)
plt.show()

## 9. Aggregate bar chart

In [ ]:
mse_mbbr_arr = np.array([r['mse_mbbr'] for r in all_results])
mse_effr_arr = np.array([r['mse_effr'] for r in all_results])
mse_neur_arr = np.array([r['mse_neur'] for r in all_results])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Mean MSE
ax = axes[0]
means = [mse_mbbr_arr.mean(), mse_effr_arr.mean(), mse_neur_arr.mean()]
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax.bar(['MBBr', 'EffR', 'Neumann'], means, color=colors)
ax.set_ylabel('Mean MSE')
ax.set_title('Average MSE across all configs')
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:.5f}', ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Median MSE
ax = axes[1]
medians = [np.median(mse_mbbr_arr), np.median(mse_effr_arr), np.median(mse_neur_arr)]
bars = ax.bar(['MBBr', 'EffR', 'Neumann'], medians, color=colors)
ax.set_ylabel('Median MSE')
ax.set_title('Median MSE across all configs')
for bar, val in zip(bars, medians):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:.5f}', ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Win counts
ax = axes[2]
n = len(all_results)
beat_effr = np.sum(mse_neur_arr <= mse_effr_arr)
beat_mbbr = np.sum(mse_neur_arr <= mse_mbbr_arr)
beat_both = np.sum((mse_neur_arr <= mse_mbbr_arr) & (mse_neur_arr <= mse_effr_arr))
bars = ax.bar(
    ['vs EffR', 'vs MBBr', 'vs Both'],
    [beat_effr, beat_mbbr, beat_both],
    color=['#FF9800', '#2196F3', '#4CAF50'],
)
ax.set_ylabel('Configs where Neumann wins')
ax.set_title(f'Neumann win counts (out of {n})')
ax.set_ylim(0, n)
ax.axhline(n, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars, [beat_effr, beat_mbbr, beat_both]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val}/{n}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Scatter: Neumann MSE vs baselines

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, baseline, name, color in [
    (axes[0], mse_mbbr_arr, 'MBBr', '#2196F3'),
    (axes[1], mse_effr_arr, 'EffR', '#FF9800'),
]:
    ax.scatter(baseline, mse_neur_arr, alpha=0.6, s=30, c=color, edgecolors='k', linewidth=0.5)
    lim_max = max(baseline.max(), mse_neur_arr.max()) * 1.1
    ax.plot([0, lim_max], [0, lim_max], 'k--', alpha=0.4, label='y=x')
    ax.set_xlabel(f'{name} MSE')
    ax.set_ylabel('Neumann MSE')
    ax.set_title(f'Neumann vs {name}')
    ax.set_xlim(0, lim_max)
    ax.set_ylim(0, lim_max)
    ax.legend()
    ax.grid(alpha=0.3)

    # Count points below/above diagonal
    below = np.sum(mse_neur_arr < baseline)
    above = np.sum(mse_neur_arr > baseline)
    ax.text(0.05, 0.92, f'Neumann better: {below}/{len(baseline)}',
            transform=ax.transAxes, fontsize=9, color='green')
    ax.text(0.05, 0.85, f'{name} better: {above}/{len(baseline)}',
            transform=ax.transAxes, fontsize=9, color='red')

plt.tight_layout()
plt.show()

## 11. Edge retention vs performance

In [ ]:
retentions = np.array([r['retention'] for r in all_results]) * 100
ratios_mbbr = mse_neur_arr / np.maximum(mse_mbbr_arr, 1e-10)
ratios_effr = mse_neur_arr / np.maximum(mse_effr_arr, 1e-10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(retentions, ratios_mbbr, alpha=0.6, s=30, c='#2196F3',
           edgecolors='k', linewidth=0.5, label='vs MBBr')
ax.scatter(retentions, ratios_effr, alpha=0.6, s=30, c='#FF9800',
           edgecolors='k', linewidth=0.5, label='vs EffR')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='parity')
ax.set_xlabel('Edge retention (%)')
ax.set_ylabel('MSE ratio (Neumann / baseline)')
ax.set_title('Neumann performance vs edge retention')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, max(ratios_mbbr.max(), ratios_effr.max()) * 1.1)
plt.tight_layout()
plt.show()